# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All elements (record sets, fields, columns) are referenced by their `@id` values to ensure unambiguous referencing.

In [ ]:
# List all record sets with their @id and name
print("Available Record Sets:")
for record_set in dataset.metadata.record_sets:
    print(f"@id: {record_set.id} | name: {getattr(record_set, 'name', '')}")
    if hasattr(record_set, 'fields'):
        print("  Fields/Columns:")
        for field in record_set.fields:
            field_id = getattr(field, 'id', None)
            field_name = getattr(field, 'name', '')
            field_type = getattr(field, 'data_type', '')
            print(f"    @id: {field_id}, name: {field_name}, type: {field_type}")
    print()

# Show at least one record from each record set by @id
for record_set in dataset.metadata.record_sets:
    print(f"Sample record from record set '@id': {record_set.id}")
    for rec in dataset.records(record_set=record_set.id):
        print(rec)
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we create a dictionary of DataFrames, one for each record set by `@id`.

In [ ]:
# Collect all record set @id's into a list
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# For demonstration, select the first record set for further analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Fields/columns for record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

All column references use their field or column `@id` where possible.

In [ ]:
# Pick a numeric field @id for analysis - example: 'cr:field:MW_kDa' (mass/weight), adjust as per actual field list
df = dataframes.get(main_record_set_id, pd.DataFrame())

numeric_field_id = None
group_field_id = None
# Attempt to auto-detect a numeric field and a groupable field
if not df.empty:
    for col in df.columns:
        # Try parsing a sample value to float
        try:
            _ = pd.to_numeric(df[col].dropna().iloc[0])
            numeric_field_id = col
            break
        except Exception:
            continue

    # Try to find a non-numeric categoric field
    for col in df.columns:
        if col != numeric_field_id:
            if df[col].dtype == object:
                group_field_id = col
                break

if numeric_field_id is None:
    print("No numeric field detected in the main record set.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")

    # Filter where numeric value > threshold (example: threshold 10)
    threshold = 10
    filtered_df = df.copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} found")
    display(filtered_df.head(3))

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(5))

    # Group by the group_field_id if found
    if group_field_id is not None:
        print(f"Grouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head(5))

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped_df exists (from previous cell):
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed information on protein abundance and properties in EVs from human mast cells, as described by the Croissant schema.
- Using `mlcroissant`, we loaded and inspected the full metadata with explicit reference to entities by `@id`.
- A numeric field was selected for filtering and normalization, illustrating approaches for downstream statistical analysis.
- Simple visualizations highlighted data distributions and group behaviors for further scientific exploration.

This notebook provides a reproducible workflow for FAIR^2 Croissant-structured datasets using the `mlcroissant` Python package.